# Phase III Part B: LightGBM Ranking Strategy
Trains a LightGBM lambdarank model on the feature table using walk-forward validation.
At each month end, ranks the 5 ETFs and allocates using a combination of normalized LightGBM scores and risk parity weights.

Walk-forward windows: 3 years train, 1 year test, roll 1 year forward.

In [1]:
import ast
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

ETFS       = ['SPY', 'EFA', 'IEF', 'VNQ', 'DBC']
MACRO_COLS = ['FEDFUNDS', 'CPIAUCSL', 'T10Y2Y', 'INDPRO']
TRAIN_MONTHS = 36  # 3 years
TEST_MONTHS  = 12  # 1 year

## Step 1: Load and Parse Feature Table

In [2]:
raw = pd.read_csv('../feature_engineer_csv.csv', parse_dates=['date'], index_col='date')

# Parse bollinger_bands dict strings into separate upper/middle/lower columns
def parse_bollinger(df, ticker):
    col = f'{ticker}_bollinger_bands'
    parsed = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else {})
    df[f'{ticker}_bb_upper']  = parsed.apply(lambda d: d.get('upper', np.nan))
    df[f'{ticker}_bb_middle'] = parsed.apply(lambda d: d.get('middle', np.nan))
    df[f'{ticker}_bb_lower']  = parsed.apply(lambda d: d.get('lower', np.nan))
    df = df.drop(columns=[col])
    return df

features = raw.copy()
for etf in ETFS:
    features = parse_bollinger(features, etf)

# Drop spoofed OHLCV columns (high/low/open/volume are identical to close)
drop_cols = [f'{etf}_{c}' for etf in ETFS for c in ['high', 'low', 'open', 'volume']]
features = features.drop(columns=drop_cols)

## Step 2: Convert to Long Format and Add Labels
Each row = one ETF at one month end.
Label = rank of ETF by actual next-month return (4 = best, 0 = worst).

In [3]:
# Load prices to compute actual next-month returns (for labelling only)
prices = pd.read_csv('../cleaned_prices.csv', index_col=0, parse_dates=True)
next_month_returns = prices[ETFS].pct_change().shift(-1)  # return EARNED next month

# Build long-format feature table
per_etf_feature_cols = ['close', 'rsi', 'macd', 'sma', 'dist_from_sma',
                         'bb_upper', 'bb_middle', 'bb_lower',
                         'dist_from_10m_sma', 'above_10m_sma']

rows = []
for date in features.index:
    if date not in next_month_returns.index:
        continue
    macro_vals = features.loc[date, MACRO_COLS].to_dict()
    next_rets   = next_month_returns.loc[date]

    for etf in ETFS:
        etf_feats = {col: features.loc[date, f'{etf}_{col}']
                     for col in per_etf_feature_cols
                     if f'{etf}_{col}' in features.columns}
        row = {'date': date, 'ticker': etf, 'next_return': next_rets[etf]}
        row.update(etf_feats)
        row.update(macro_vals)
        rows.append(row)

long_df = pd.DataFrame(rows).set_index('date').sort_index()

# Rank ETFs within each month: 4 = best next-month return, 0 = worst
long_df['label'] = long_df.groupby('date')['next_return'].rank(method='first').astype(int) - 1

# Drop NaN rows (last month has no next_return)
long_df = long_df.dropna(subset=['next_return'])

long_df.head(10)

,ticker,next_return,close,rsi,macd,sma,dist_from_sma,bb_upper,bb_middle,bb_lower,dist_from_10m_sma,above_10m_sma,FEDFUNDS,CPIAUCSL,T10Y2Y,INDPRO,label
date,,,,,,,,,,,,,,,,,
2007-09-30,SPY,0.013566,108.383331,75.091606,NaN,97.266300,0.114295,111.091612,97.266300,83.440988,0.048917,1,4.94,208.292,0.62,114.3570,1
2007-09-30,EFA,0.042499,46.893768,82.834171,NaN,40.742856,0.150969,48.426662,40.742856,33.059050,0.060478,1,4.94,208.292,0.62,114.3570,3
2007-09-30,IEF,0.010769,52.976456,72.067480,NaN,50.256057,0.054131,53.340075,50.256057,47.172039,0.030467,1,4.94,208.292,0.62,114.3570,0
2007-09-30,VNQ,0.020991,32.989182,59.080627,NaN,32.316259,0.020823,38.465251,32.316259,26.167266,-0.037596,0,4.94,208.292,0.62,114.3570,2
2007-09-30,DBC,0.085023,22.830553,72.026599,NaN,20.241358,0.127916,22.121983,20.241358,18.360734,0.098347,1,4.94,208.292,0.62,114.3570,4
2007-10-31,SPY,-0.038732,109.853683,76.353775,NaN,98.345065,0.117023,112.558308,98.345065,84.131821,0.052398,1,4.76,208.903,0.54,113.9957,1
2007-10-31,EFA,-0.036237,48.886703,85.172442,NaN,41.451770,0.179363,49.389792,41.451770,33.513748,0.087560,1,4.76,208.903,0.54,113.9957,2
2007-10-31,IEF,0.040477,53.546986,74.210038,NaN,50.490176,0.060543,53.817447,50.490176,47.162906,0.035430,1,4.76,208.903,0.54,113.9957,4
2007-10-31,VNQ,-0.094709,33.681648,60.603147,NaN,32.601693,0.033126,38.439863,32.601693,26.763524,-0.014908,0,4.76,208.903,0.54,113.9957,0


## Step 3: Walk-Forward Validation + LightGBM Training

In [4]:
feature_cols = [c for c in long_df.columns if c not in ['ticker', 'next_return', 'label']]

dates      = sorted(long_df.index.unique())
n_dates    = len(dates)
all_preds  = []

i = 0
while i + TRAIN_MONTHS + TEST_MONTHS <= n_dates:
    train_dates = dates[i : i + TRAIN_MONTHS]
    test_dates  = dates[i + TRAIN_MONTHS : i + TRAIN_MONTHS + TEST_MONTHS]

    # Drop rows where MACD is NaN rather than filling with 0
    # MACD needs ~26 months of history; early rows are genuinely missing, not zero
    train_df = long_df[long_df.index.isin(train_dates)].dropna(subset=['macd'])
    test_df  = long_df[long_df.index.isin(test_dates)]

    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df['label']
    groups  = train_df.groupby('date').size().values  # recomputed after dropna

    X_test  = test_df[feature_cols].fillna(0)

    train_data = lgb.Dataset(X_train, label=y_train, group=groups)

    params = {
        'objective':    'lambdarank',
        'metric':       'ndcg',
        'learning_rate': 0.05,
        'num_leaves':   31,
        'verbose':      -1,
    }

    model = lgb.train(params, train_data, num_boost_round=100)
    scores = model.predict(X_test)

    for j, (idx, row) in enumerate(test_df.iterrows()):
        all_preds.append({'date': idx, 'ticker': row['ticker'], 'score': scores[j]})

    i += TEST_MONTHS

preds_df = pd.DataFrame(all_preds)

## Step 4: Combined Scoring and Backtest

In [5]:
# 60-day daily volatility for risk parity weights
prices_daily = pd.read_csv("../cleaned_prices_daily.csv", index_col=0, parse_dates=True)
vol_60d = prices_daily[ETFS].pct_change().rolling(60).std()

monthly_returns = prices[ETFS].pct_change()

mr_dates = sorted(monthly_returns.index)
next_date_map = {mr_dates[i]: mr_dates[i + 1] for i in range(len(mr_dates) - 1)}

COST  = 0.001 # 0.1% per unit of one-way turnover

prev_weights = pd.Series(0.0, index=ETFS)
portfolio_returns = []

for date, group in preds_df.groupby("date"):
    ret_date = next_date_map.get(date)
    if ret_date is None or ret_date not in monthly_returns.index:
        continue

    # ML scores for all 5 ETFs → linear weights (shift so min=0, then normalize)
    scores = group.set_index('ticker').loc[ETFS, 'score']
    shifted = scores - scores.min()
    score_weights = shifted / shifted.sum()

    # Risk parity weights for all 5 ETFs
    vols = vol_60d.asof(date)[ETFS].replace(0, np.nan)
    inv_vol = (1.0 / vols).fillna(0)
    if inv_vol.sum() == 0:
        continue
    rp_weights = inv_vol / inv_vol.sum()

    # Combined: ML signal × risk parity, renormalized
    combined = score_weights * rp_weights
    weights = combined / combined.sum()

    turnover = (weights - prev_weights).abs().sum() / 2
    txn_cost = COST * turnover

    ret = (monthly_returns.loc[ret_date, ETFS] * weights).sum() - txn_cost
    portfolio_returns.append({"date": ret_date, "return": ret, "n_invested": 5})
    prev_weights = weights

ml_returns = pd.DataFrame(portfolio_returns).set_index("date")
ml_returns.head(10)

,return,n_invested
date,,
2010-10-31,0.016562,5
2010-11-30,-0.013504,5
2010-12-31,0.012300,5
2011-01-31,0.011280,5
2011-02-28,0.019362,5
2011-03-31,-0.006294,5
2011-04-30,0.027816,5
2011-05-31,0.004010,5
2011-06-30,-0.010186,5


## Step 5: Save Results

In [6]:
ml_returns.to_csv('../ml_returns.csv')